## **LangChain Deep Technical Implementation using HuggingFace API**

This notebook demonstrates the implementation of LangChain components using HuggingFace Inference API. It covers LLM interaction, prompt templates, chains, memory, agents, document loaders, and vector stores.

## 1. Installing Required Libraries

# In this step, we install all necessary libraries including LangChain and HuggingFace Hub for API interaction.

In [ ]:
!pip install -U langchain langchain-core langchain-community langchain-huggingface huggingface_hub

In [ ]:
import os

from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

## 2. Setting up HuggingFace API

We initialise the HuggingFace Inference Client using an API token. This allows us to interact with hosted models.

In [ ]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "Token"

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id="google/flan-t5-base"
    task="text2text-generation"
    temperature=0.7,
    max_new_tokens=100
)

## 3. Basic LLM Call

This section demonstrates how to send a simple prompt to the HuggingFace model and receive a response.

In [ ]:
print(llm.invoke("Explain LangChain in simple terms"))

## 4. LangChain Integration

Here, we wrap the HuggingFace API inside a LangChain Runnable to make it compatible with LangChain pipelines.

In [ ]:
template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms"
)

print(template.format(topic="Machine Learning"))

## 5. Chains using LCEL

LangChain Expression Language (LCEL) allows us to connect prompts, models, and output parsers into a pipeline.

In [ ]:
chain = template | llm | StrOutputParser()

print(chain.invoke({"topic": "Artificial Intelligence"}))

## 6. Memory Implementation

In this section, we simulate conversational memory by storing previous interactions and passing them to the model.

In [ ]:
chat_history = []

def chat(user_input):
    chat_history.append(user_input)

    prompt = f"""
You are a helpful assistant.

Conversation:
{chr(10).join(chat_history)}

Question: {user_input}
Answer:
"""

    response = llm.invoke(prompt)

    chat_history.append(response)

    return response


print(chat("Hi, my name is Arjun"))
print(chat("What is my name?"))

## 7. Tools and Agents

Agents use tools to perform specific tasks. Here, we create a simple calculator tool and integrate it with an agent.

In [ ]:
from langchain.tools import Tool
from langchain.agents import initialize_agent

def calculator(query):
    try:
        return str(eval(query))
    except:
        return "Error"

tools = [
    Tool(
        name="Calculator",
        func=calculator,
        description="Performs math calculations"
    )
]

agent = initialize_agent(
    tools,
    llm,
    agent="zero-shot-react-description",
    verbose=True
)

print(agent.run("What is 20 + 5?"))

## Document Loader

This section demonstrates how to load external text data using LangChain's document loader.

In [ ]:
from langchain_community.document_loaders import TextLoader

# Create a sample file
with open("sample.txt", "w") as f:
    f.write("LangChain helps in building powerful LLM applications using modular components.")

# Load the document
loader = TextLoader("sample.txt")
documents = loader.load()

# Print output
print(documents)

## Vector Store (FAISS)

This section demonstrates how to store text as embeddings and perform similarity search using FAISS.

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Initialize embedding model
embedding = HuggingFaceEmbeddings()

# Sample texts
texts = [
    "LangChain is used for LLM applications",
    "Artificial Intelligence is transforming industries",
    "Machine learning enables data-driven decisions"
]

# Create vector store
vectorstore = FAISS.from_texts(texts, embedding)

# Perform similarity search
query = "What is AI?"
results = vectorstore.similarity_search(query)

# Print results
for i, doc in enumerate(results):
    print(f"Result {i+1}: {doc.page_content}")